In [1]:
!pip install langchain langchain-community langchain-core langchain-google-genai langchain-text-splitters faiss-cpu pypdf tiktoken

In [2]:
import os, re, time, getpass
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [4]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("/content/intro-to-ml-book.pdf").load()
print("Pages loaded:", len(pages))

Pages loaded: 392


In [5]:
all_text = "\n".join(p.page_content for p in pages)

print("Number of pages:", len(pages))
print("Empty pages (under 20 characters):", sum(1 for p in pages if len(p.page_content.strip()) < 20))
print("Words broken across lines by a hyphen:", len(re.findall(r"\w-\n\w", all_text)))
print("Page footers such as 'Title | 37':", len(re.findall(r"\|\s*\d+\s*$", all_text, flags=re.M)))
print("Book title repeated on pages:", all_text.count("Introduction to Machine Learning with Python"))

Number of pages: 392
Empty pages (under 20 characters): 5
Words broken across lines by a hyphen: 56
Page footers such as 'Title | 37': 179
Book title repeated on pages: 2


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=700,
    chunk_overlap=100,
)
chunks = splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 410


In [7]:
import time
from langchain_community.vectorstores import FAISS

vector_store = None
BATCH_SIZE = 10  # 10 requests per batch (~11,500 tokens max, well under 30,000 TPM)

for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]

    if vector_store is None:
        vector_store = FAISS.from_documents(batch, embedding)
    else:
        vector_store.add_documents(batch)

    print(f"Embedded {min(start + BATCH_SIZE, len(chunks))} of {len(chunks)} chunks")

    # Sleep 62s between batches so you never hit TPM/RPM limits and never waste quota on retries
    if start + BATCH_SIZE < len(chunks):
        time.sleep(62)

# Save locally so you never have to re-embed these chunks again!
vector_store.save_local("faiss_index")
print("Saved FAISS index! Total vectors:", vector_store.index.ntotal)

Embedded 10 of 410 chunks
Embedded 20 of 410 chunks
Embedded 30 of 410 chunks
Embedded 40 of 410 chunks
Embedded 50 of 410 chunks
Embedded 60 of 410 chunks
Embedded 70 of 410 chunks
Embedded 80 of 410 chunks
Embedded 90 of 410 chunks
Embedded 100 of 410 chunks
Embedded 110 of 410 chunks
Embedded 120 of 410 chunks
Embedded 130 of 410 chunks
Embedded 140 of 410 chunks
Embedded 150 of 410 chunks
Embedded 160 of 410 chunks
Embedded 170 of 410 chunks
Embedded 180 of 410 chunks
Embedded 190 of 410 chunks
Embedded 200 of 410 chunks
Embedded 210 of 410 chunks
Embedded 220 of 410 chunks
Embedded 230 of 410 chunks
Embedded 240 of 410 chunks
Embedded 250 of 410 chunks
Embedded 260 of 410 chunks
Embedded 270 of 410 chunks
Embedded 280 of 410 chunks
Embedded 290 of 410 chunks
Embedded 300 of 410 chunks
Embedded 310 of 410 chunks
Embedded 320 of 410 chunks
Embedded 330 of 410 chunks
Embedded 340 of 410 chunks
Embedded 350 of 410 chunks
Embedded 360 of 410 chunks
Embedded 370 of 410 chunks
Embedded 3

In [8]:
sample_query = "What is overfitting?"

query_vector = embedding.embed_query(sample_query)
results = vector_store.similarity_search_by_vector(query_vector, k=3)

for doc in results:
    print("Page", doc.metadata["page"] + 1)
    print(doc.page_content[:300], "...")
    print()

Page 42
3 And also provably, with the right math.
58 years old want to buy a boat, while all others don’t. While we can make up many
rules that work well on this data, remember that we are not interested in making pre‐
dictions for this dataset; we already know the answers for these customers. We want
to kn ...

Page 41
2 In the real world, this is actually a tricky problem. While we know that the other customers haven’t bought a
boat from us yet, they might have bought one from someone else, or they may still be saving and plan to buy
one in the future.
Usually we build a model in such a way that it can make accur ...

Page 62
You might notice the strange-looking trailing underscore at the end
of coef_ and intercept_. scikit-learn always stores anything
that is derived from the training data in attributes that end with a
trailing underscore. That is to separate them from parameters that
are set by the user.
The intercept_ ...



In [9]:
def retrieve(question, k=5):
    query_vector = embedding.embed_query(question)            # query -> embedding
    return vector_store.similarity_search_by_vector(query_vector, k=k)   # top-k chunks

In [10]:
experiment_query = "What is overfitting?"

for k in [3, 5, 7]:
    docs = retrieve(experiment_query, k=k)
    pages_found = sorted({d.metadata["page"] + 1 for d in docs})
    print(f"k={k}: {len(docs)} chunks, from pages {pages_found}")

k=3: 3 chunks, from pages [41, 42, 62]
k=5: 5 chunks, from pages [41, 42, 43, 53, 62]
k=7: 7 chunks, from pages [41, 42, 43, 53, 62, 63, 73]


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a machine learning tutor.
Answer the question using ONLY the context below. Do not use any other knowledge.
If the answer is not in the context, reply exactly: "I couldn't find this in the book."
After each point, cite the page number, for example (p. 42).

Context:
{context}

Question: {question}
""")

chain = prompt | llm | StrOutputParser()

In [12]:
def answer(question, k=5):
    docs = retrieve(question, k=k)
    context = "\n\n".join(f"[p. {d.metadata['page'] + 1}] {d.page_content}" for d in docs)
    response = chain.invoke({"context": context, "question": question})

    print("Question:", question)
    print("Answer:", response)
    print("Pages used:", sorted({d.metadata["page"] + 1 for d in docs}))
    print("\nRetrieved context:")
    for d in docs:
        print(f"  [p. {d.metadata['page'] + 1}] {d.page_content[:150]}...")
    print("-" * 80)

In [13]:
demo_questions = [
    "What is overfitting and how can we detect it?",
    "How does the k-nearest neighbors algorithm make a prediction?",
    "What does one-hot encoding do?",
    "What is cross-validation?",
    "What is the capital of France?",
]

for q in demo_questions:
    answer(q, k=5)

Question: What is overfitting and how can we detect it?
Answer: Based on the provided text, here is what overfitting is and how it can be detected:

**What Overfitting Is:**
* Overfitting means building a model that is too complex for the amount of information available (p. 42).
* It occurs when a model is fit too closely to the particularities of the training set by focusing too much on each individual data point, resulting in a model that performs well on the training set but cannot generalize to new data (p. 42).

**How to Detect It:**
* The only measure of whether an algorithm will perform well on new data is evaluation on the test set (p. 42).
* It can be detected by comparing the model's performance on the training set versus the test set: overfitting is indicated when the model predicts very accurately (or achieves high/perfect accuracy) on the training set, but performs significantly worse on the test set (p. 53, p. 62).
Pages used: [41, 42, 53, 62, 73]

Retrieved context:
  [p

In [14]:
while True:
    question = input("Your question: ")
    if question.strip().lower() == "exit":
        print("Goodbye!")
        break
    answer(question, k=5)

Your question: What is the capital of Kerala?
Question: What is the capital of Kerala?
Answer: I couldn't find this in the book.
Pages used: [1, 3, 27, 31, 392]

Retrieved context:
  [p. 392] activity near their habitat means greater amounts of sediment and chemicals in the
water. In an effort to save this endangered species, biologists hav...
  [p. 1] Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y thon   
A GUIDE FOR DATA SCIENTISTS...
  [p. 3] Andreas C. Müller and Sarah Guido
Introduction to Machine Learning
with Python
A Guide for Data Scientists
Boston Farnham Sebastopol TokyoBeijing Bost...
  [p. 27] Out[9]:
Python version: 3.5.2 |Anaconda 4.1.1 (64-bit)| (default, Jul  2 2016, 17:53:06)
[GCC 4.4.7 20120313 (Red Hat 4.4.7-1)]
pandas version: 0.18.1...
  [p. 31] In[19]:
print("Shape of target: {}".format(iris_dataset['target'].shape))
Out[19]:
Shape of target: (150,)
The species are encoded as integers from 0 ...
-------------------------------------